# PPP ensemble analysis

Analysis of the reduced ensembles written by `Executables/PPP/python/ppp_reduce.py` (one file per Markov chain,
`<run>_cNN.h5`):

1. Monte Carlo history
2. autocorrelation times and the bin size
3. thermalization and agreement between chains
4. ergodicity: sign of the fermion determinant
5. single-particle spectrum (electron addition energies; removal energies too without particle-hole symmetry;
   charge gap) and correlator matrices
6. equal-time spin and charge correlations
7. heavy tails of the distributions (where error estimates are unreliable)
8. two-body correlators: singlet and triplet excitons, pp pairs (stage 2, `ppp_reduce2.py`): spectrum,
   correlator matrices, heavy tails, pseudospin test
9. level diagram: non-interacting and interacting energies, one- and two-body

All computations are in `ppp_analysis.py` (tests: `test_ppp_analysis.py`); the Wick contractions are in
`ppp_wick.py` (tested against exact Fock-space traces by `test_ppp_wick.py`).

Units: energies are given as the ratio $E/\kappa$ with the hopping parameter $\kappa$. Imaginary time is
$\tau = \delta\, t$ with the time slice $t = 0, \dots, N_t - 1$ and $\delta = \beta/N_t$, so $\tau$ runs from $0$ to
$\beta$ (in units of $1/\kappa$, like $\beta$ in the input file). Settings that count time slices (`T0`, `TD`,
`FIT_WINDOW`, `T_PROBE`) use $t$.

Errors: jackknife over bins of `BINSIZE` consecutive configurations, with the bins of all chains of a run pooled.

In [ ]:
# --- settings ------------------------------------------------------------------------------------
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "python")))   # Executables/PPP/python

DATA_DIR = os.environ.get("PPP_REDUCED", "~/Research/perylene_b32/reduced")       # the <run>_cNN.h5 files
RUN_ORDER = ["perylene_ohno_b32_Nt128", "perylene_mataga-nishimoto_b32_Nt128", "perylene_hubbard_b32_Nt128"]
LABELS = {"perylene_ohno_b32_Nt128": "Ohno",
          "perylene_mataga-nishimoto_b32_Nt128": "Mataga-Nishimoto",
          "perylene_hubbard_b32_Nt128": "Hubbard"}
DISCARD = 0              # configurations dropped at the start of every chain (a number, or {run: number})
BINSIZE = 100            # configurations per bin; a multiple of the bins of the reduced files (--bin); check in section 2
T0, TD = 2, 6            # GEVP: C(TD) v = lambda C(T0) v (time slices)
FIT_WINDOW = (8, 48)     # time slices of the single-exponential fits of the GEVP states, cut for every state
MAX_REL_ERR = 0.25       # where the relative error of its correlator exceeds MAX_REL_ERR
FIT_OVERRIDES = {}       # explicit windows: {(run, class, state): (first, last time slice)}; removal fits
                         # (runs without particle-hole symmetry): key (run, class, state, "removal")
T_PROBE = 8              # time slice of the correlator used for autocorrelation, chain checks and tails
                         # (files reduced with --bin: one of the probe slices, by default 4, 8, 16, 32)
CORR_MAX_OPS = 6         # correlator-matrix figures: at most this many operators (orbitals, pairs) per block
LEVEL_MAX_ERR = 0.3      # level diagram: interacting levels with a larger error (in E/kappa) are left out

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import ppp_analysis as pa

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8a85",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.5, "lines.markersize": 4, "errorbar.capsize": 0,
    "legend.frameon": False, "text.color": "#2b2b29", "axes.labelcolor": "#2b2b29",
    "xtick.color": "#5c5c57", "ytick.color": "#5c5c57"})
pd.set_option("display.precision", 4)
TAU = r"$\tau$"                    # imaginary time delta * t, from 0 to beta
EKAPPA = r"$E_{\rm eff}/\kappa$"   # effective energy over the hopping

runs = pa.load_runs(DATA_DIR, RUN_ORDER, DISCARD)
assert runs, f"no reduced files in {DATA_DIR}"
label = lambda r: LABELS.get(r, r)
color = {r: pa.COLORS[i] for i, r in enumerate(runs)}

def ppp_block(run):
    return yaml.safe_load(run.yml_text).get("PPP", {})

rows = []
for name, run in runs.items():
    p, c0 = ppp_block(run), run.chains[0]
    per = sorted({c.n for c in run.chains})
    rows.append({"run": label(name), "chains": len(run.chains), "configurations": run.n,
                 "per chain": str(per[0]) if len(per) == 1 else f"{per[0]}-{per[-1]}",
                 "stored bins": max(c.bin for c in run.chains), "Nt": run.Nt, "beta": run.beta, "delta": run.delta,
                 "interaction": p.get("form", "on-site"), "U/kappa": run.V[0, 0],
                 "sources": ", ".join(map(str, c0.sources))})
pd.DataFrame(rows).set_index("run")

## 1. Monte Carlo history

The action $S$ and the squared amplitude $a_0^2$ of the time-summed field along the eigenvector of $V$ with the
largest eigenvalue (the uniform charge mode, the slowest mode in earlier runs). The chains of a run are shown one
after the other (separated by vertical lines); light: every configuration, dark: averages over bins of `BINSIZE`.
A drift at the start of a chain means incomplete thermalization (set `DISCARD`); a chain that sits at a different
level from the others is a slowly decorrelating region (sections 2 and 3).

In [ ]:
fig, axes = plt.subplots(len(runs), 2, figsize=(11, 2.4 * len(runs)), squeeze=False, constrained_layout=True)
for i, (name, run) in enumerate(runs.items()):
    start = 0
    for k, c in enumerate(run.chains):
        x = start + np.arange(c.n)
        xb = pa.bin_mean(x, BINSIZE)
        for ax, y in zip(axes[i], (c.S, c.field_modes[:, 0] ** 2)):
            ax.plot(x, y, lw=0.4, color=color[name], alpha=0.3)
            ax.plot(xb, pa.bin_mean(y, BINSIZE), lw=1.2, color=color[name])
        start += c.n
        if k < len(run.chains) - 1:
            for ax in axes[i]:
                ax.axvline(start, color="#8a8a85", lw=0.4, alpha=0.6)
    axes[i, 0].set_title(f"{label(name)}: action S ({len(run.chains)} chains)")
    axes[i, 1].set_title(f"{label(name)}: uniform charge mode $a_0^2$")
    axes[i, 0].set_ylabel("$S$")
    axes[i, 1].set_ylabel("$a_0^2$")
for ax in axes[-1]:
    ax.set_xlabel("configuration (chains one after the other)")
plt.show()

## 2. Autocorrelation times and bin size

$\tau_{\rm int}$ in units of configurations (Gamma method), for the action, its fermionic part, $a_0^2$, and the
addition correlator of the LUMO at time slice `T_PROBE` (with particle-hole symmetry, the removal correlator of the
HOMO is identical on every configuration, see section 5). The table gives the mean over the chains of a run (error: spread of the chains) and
the largest chain value. The plot shows $\tau_{\rm int}$ from the variance of bin averages as a function of the bin
size, averaged over the chains: it saturates once the bins are independent; rare long excursions show up there and
not in the windowed Gamma method. Bins of `BINSIZE` configurations should be several times the largest
$\tau_{\rm int}$.

In [ ]:
def lumo(run):
    # (class, index in the class block) of the LUMO
    o = int(np.argmin(np.where(run.eps > 0, run.eps, np.inf)))
    cls = run.orbital_class[o]
    return cls, int(np.nonzero(run.classes[cls] == o)[0][0])

PROBE_KEY = f"A_LUMO(t={T_PROBE})"

def history(c, run):
    return {"S": c.S, "S_fermion": c.S_fermion, "a_0^2": c.field_modes[:, 0] ** 2,
            PROBE_KEY: c.onebody_element(*lumo(run), T_PROBE, "addition")}

def over_chains(values, tags):
    v = np.asarray(values, float)
    k = int(np.argmax(v))
    spread = f" ± {v.std(ddof=1) / np.sqrt(len(v)):.2f}" if len(v) > 1 else ""
    return f"{v.mean():.2f}{spread}  (max {v[k]:.2f}, {tags[k]})"

histories = {name: [history(c, run) for c in run.chains] for name, run in runs.items()}
rows = []
for name, run in runs.items():
    tags = [c.tag[-3:] for c in run.chains]
    row = {"run": label(name), "chains": len(run.chains),
           "acceptance": over_chains([np.mean(np.minimum(c.acceptance, 1)) for c in run.chains], tags)}
    for key in histories[name][0]:
        row["tau_int " + key] = over_chains([pa.tau_int(h[key])[0] for h in histories[name]], tags)
    rows.append(row)
display(pd.DataFrame(rows).set_index("run").T)

binsizes = 2 ** np.arange(0, 10)
keys = [("S_fermion", r"$S_{\rm fermion}$"), ("a_0^2", r"$a_0^2$"), (PROBE_KEY, f"$A_{{\\rm LUMO}}(t={T_PROBE})$")]
fig, axes = plt.subplots(1, len(keys), figsize=(12, 3), constrained_layout=True)
for ax, (key, tex) in zip(axes, keys):
    for name, run in runs.items():
        n_min = min(len(h[key]) for h in histories[name])
        b = binsizes[binsizes <= n_min // 10]
        curves = np.array([pa.tau_int_binned(h[key], b) for h in histories[name]])
        err = curves.std(axis=0, ddof=1) / np.sqrt(len(curves)) if len(curves) > 1 else None
        ax.errorbar(b, curves.mean(axis=0), err, fmt="o-", ms=3, color=color[name], label=label(name))
    ax.axvline(BINSIZE, color="#8a8a85", lw=1, ls=":")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("bin size (configurations)")
    ax.set_title(tex + " (mean over chains)")
axes[0].set_ylabel(r"$\tau_{\rm int}$ from bin averages (configurations)")
axes[0].legend()
plt.show()

## 3. Thermalization and agreement between chains

For every chain: the pull of its mean from the weighted mean of the run, and the difference between its second
and first half, each in units of its error (errors from bins of `BINSIZE`). The table gives the run mean,
$\chi^2/{\rm dof}$ of the chain means about it, and the number of chains whose halves differ by more than
$3\sigma$. Differences of many standard deviations, or $\chi^2/{\rm dof} \gg 1$, mean incomplete thermalization
(first half) or regions of configuration space that one chain visits and another does not; the spread between
chains is then the more honest error. With many chains, about 0.3 % of the pulls exceed $3\sigma$ by chance.

In [ ]:
rows, pulls = [], {}
keys = list(next(iter(histories.values()))[0])
tex = {"S": "$S$", "S_fermion": r"$S_{\rm fermion}$", "a_0^2": "$a_0^2$", PROBE_KEY: f"$A_{{\\rm LUMO}}(t={T_PROBE})$"}
for name, run in runs.items():
    for k in keys:
        ms, es, halves = [], [], []
        for h in histories[name]:
            x, half = h[k], len(h[k]) // 2
            m, e = pa.mean_error(x, BINSIZE)
            m1, e1 = pa.mean_error(x[:half], BINSIZE)
            m2, e2 = pa.mean_error(x[half:], BINSIZE)
            ms.append(m); es.append(e); halves.append((m2 - m1) / np.hypot(e1, e2))
        ms, es, halves = np.array(ms), np.array(es), np.array(halves)
        wm, we, chi2, _ = pa.consistency(ms, es)
        pulls[(name, k)] = ((ms - wm) / es, halves)
        rows.append({"run": label(name), "observable": k, "mean": wm, "error": we, "chi2/dof of the chain means": chi2,
                     "chains with mean > 3σ off": f"{int(np.sum(np.abs((ms - wm) / es) > 3))}/{len(ms)}",
                     "chains with halves > 3σ apart": f"{int(np.sum(np.abs(halves) > 3))}/{len(ms)}"})
display(pd.DataFrame(rows).set_index(["run", "observable"]))

fig, axes = plt.subplots(2, len(keys), figsize=(3.1 * len(keys), 5), sharex=True, constrained_layout=True)
for j, k in enumerate(keys):
    for name, run in runs.items():
        p, hv = pulls[(name, k)]
        idx = np.arange(1, len(p) + 1)
        axes[0, j].plot(idx, p, "o", ms=3, color=color[name], label=label(name))
        axes[1, j].plot(idx, hv, "o", ms=3, color=color[name])
    for ax in axes[:, j]:
        for y in (-3, 3):
            ax.axhline(y, color="#8a8a85", lw=0.8, ls="--")
        ax.axhline(0, color="#8a8a85", lw=0.6)
    axes[0, j].set_title(tex[k])
    axes[1, j].set_xlabel("chain")
axes[0, 0].set_ylabel("chain mean - run mean (σ)")
axes[1, 0].set_ylabel("second - first half (σ)")
try:
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside upper right", ncol=len(runs))
except ValueError:                                     # matplotlib < 3.7
    axes[0, 0].legend(fontsize=8)
plt.show()

## 4. Ergodicity: sign of the fermion determinant

At $\mu = 0$ on a bipartite lattice, $\det M_p\, e^{-i\sum\phi/2}$ is real. Its sign labels regions of
configuration space separated by zeros of the determinant, which the HMC cannot cross; the radial update can,
if its steps are large enough (radial scale $\sim 1/\sqrt{N_t N_x}$, arXiv:2504.01575; its acceptance is in the
HMC log). All chains start at $\phi = 0$, in the $+$ region. If every configuration of every chain has the same
sign, this check cannot distinguish "the other regions do not matter" from "the chains are trapped".

In [ ]:
rows, minus = [], []
for name, run in runs.items():
    cs = [pa.sign_census(c) for c in run.chains]
    rows.append({"run": label(name), "chains": len(cs), "computed": sum(c["computed"] for c in cs),
                 "+1": sum(c["plus"] for c in cs), "-1": sum(c["minus"] for c in cs),
                 "chains with -1": sum(c["minus"] > 0 for c in cs), "sign changes": sum(c["changes"] for c in cs)})
    minus += [dict(run=label(name), **c) for c in cs if c["minus"] > 0]
display(pd.DataFrame(rows).set_index("run"))
if minus:
    display(pd.DataFrame(minus).set_index(["run", "chain"]))

## 5. Single-particle spectrum

For each symmetry class, the addition correlator matrix $A_{ab}(\tau) = \langle c_a(\tau) c_b^\dagger(0)\rangle$ of
the class orbitals with $\varepsilon > 0$. The generalized eigenvalue problem $C(t_D)\,v = \lambda\, C(t_0)\,v$
on the mean gives the states; each jackknife sample is projected on them and fitted with a single exponential. The
fit window of a state is `FIT_WINDOW`, cut where the relative error of its correlator exceeds `MAX_REL_ERR`
(or `FIT_OVERRIDES`). The panels show the effective energies $E_{\rm eff}(\tau) = \log[c(\tau)/c(\tau+\delta)]/\delta$
over $\kappa$, the fits (solid, over the window) and the non-interacting orbital energies $\varepsilon$ (dashed).

**Particle-hole symmetry** ($\mu = 0$ on a bipartite lattice, `pa.particle_hole_symmetric`): particle-hole
conjugation gives $R_{vv'}(\tau) = A_{cc'}(\tau)^*$ on every configuration, for the valence orbitals $v = (-1)^x c$
(tested to $10^{-14}$ in `test_ppp_analysis.py`). The removal spectrum of a class is then the addition spectrum of its
partner class (column "removal class"), so only addition energies are shown, and the charge gap
$E_{N+1} + E_{N-1} - 2E_N$ is twice the lowest addition energy.

**Without it** ($\mu \ne 0$, or a lattice that is not bipartite) the removal correlator
$R_{ab}(\tau) = \langle c_a^\dagger(\tau) c_b(0)\rangle$ of the class orbitals with $\varepsilon < 0$ is analyzed in the
same way; its states have the energies $E_{N-1} - E_N$ (non-interacting: $-\varepsilon$), and the charge gap is the
lowest addition plus the lowest removal energy (error: jackknife of the sum, both fits use the same bins).

In [ ]:
PH = {name: pa.particle_hole_symmetric(run) for name, run in runs.items()}
for name, run in runs.items():
    print(f"{label(name)}: " + ("particle-hole symmetric (mu = 0, bipartite): addition energies only" if PH[name]
                                else f"no particle-hole symmetry (mu = {run.mu:g}"
                                     + (", not bipartite" if run.sublattice is None else "") + "): addition and removal"))

def overrides(name, cls, kind):
    return {k[2]: w for k, w in FIT_OVERRIDES.items()
            if k[0] == name and k[1] == cls and (k[3] if len(k) > 3 else "addition") == kind}

spectra, removal_spectra = {}, {}        # per run and class: GEVP results (energies E_{N+1} - E_N, E_{N-1} - E_N)
onebody_corr, onebody_free = {}, {}      # per class: [(run label, colour, mean, error)], non-interacting correlator
for name, run in runs.items():
    spectra[name], removal_spectra[name] = {}, {}
    for cls in run.classes:
        A, R = run.onebody(cls, BINSIZE)
        for kind, C, store in (("addition", A, spectra[name]), ("removal", R, removal_spectra[name])):
            if kind == "removal" and PH[name]:
                continue
            sel = pa.orbital_selection(run, cls, kind)
            res = pa.state_energies(C, run.delta, T0, TD, FIT_WINDOW, sel, MAX_REL_ERR, overrides(name, cls, kind))
            if res is None:
                continue
            eps = run.eps[run.classes[cls][sel]]
            res["free"] = np.sort(eps if kind == "addition" else -eps)
            store[cls] = res
            if kind == "addition":
                As = A[:, :, sel][:, :, :, sel]
                onebody_corr.setdefault(cls, []).append((label(name), color[name], As.mean(axis=0),
                                                         As.std(axis=0, ddof=1) / np.sqrt(len(As))))
                if run.has_noninteracting and cls not in onebody_free:
                    onebody_free[cls] = run.chains[0].noninteracting_onebody(cls)[0][:, sel][:, :, sel]

def plot_spectrum(name, run, kind, spec):
    classes = list(spec)
    fig, axes = plt.subplots(1, len(classes), figsize=(2.9 * len(classes), 3.0), sharey=True, squeeze=False,
                             constrained_layout=True)
    fig.suptitle(f"{label(name)}: {kind} energies (effective energies of the GEVP states)", x=0.01, ha="left")
    tmax = 0
    for ax, cls in zip(axes[0], classes):
        res = spec[cls]
        tau = np.arange(res["Eeff"].shape[1]) * run.delta
        for n in range(len(res["E"])):
            good = np.isfinite(res["Eeff"][n]) & (res["Eeff_err"][n] < 0.5)
            last = len(good) if good.all() else int(np.argmin(good))   # signal up to the first bad time slice
            ok = np.arange(len(good)) < last
            ax.errorbar(tau[ok], res["Eeff"][n, ok], res["Eeff_err"][n, ok], fmt="o", ms=2.5, lw=0.8,
                        color=pa.COLORS[n], label=f"state {n}")
            tmax = max(tmax, tau[last - 1] if last else 0)
            if res["windows"][n] is not None:
                ax.plot(np.array(res["windows"][n]) * run.delta, [res["E"][n]] * 2, color=pa.COLORS[n], lw=2.5)
            ax.axhline(res["free"][n], color=pa.COLORS[n], lw=0.8, ls="--")
        ax.set_title(f"class {pa.sym_label(cls)}")
        ax.set_xlabel(TAU)
    for ax in axes[0]:
        ax.set_xlim(0, 1.15 * tmax + 0.5)
        ax.set_ylim(0, 3.2)
    axes[0, 0].set_ylabel(EKAPPA)
    axes[0, 0].legend(fontsize=7)
    plt.show()

for name, run in runs.items():
    for kind, spec in (("addition", spectra[name]), ("removal", removal_spectra[name])):
        if spec:
            plot_spectrum(name, run, kind, spec)

In [ ]:
rows = []
for name, run in runs.items():
    for kind, spec in (("addition", spectra[name]), ("removal", removal_spectra[name])):
        for cls, res in spec.items():
            head = {"run": label(name), "energy": kind, "class": cls}
            if PH[name]:
                o = run.classes[cls][pa.orbital_selection(run, cls, "addition")[0]]
                head["removal class"] = run.orbital_class[run.partner[o]] if run.partner[o] >= 0 else "-"
            for n, (E, dE, w, free) in enumerate(zip(res["E"], res["dE"], res["windows"], res["free"])):
                rows.append({**head, "state": n, "E/kappa": E, "error": dE, "free (eps or -eps)/kappa": free,
                             "fit window (tau)": f"{w[0] * run.delta:.2f}-{w[1] * run.delta:.2f}" if w else "-"})
table = pd.DataFrame(rows)
if all(PH.values()):
    table = table.drop(columns="energy").rename(columns={"free (eps or -eps)/kappa": "eps/kappa (free)"})
    table = table.set_index(["run", "class", "state"])
else:
    table = table.set_index(["run", "energy", "class", "state"])
table

In [ ]:
def lowest_level(spec):
    cand = [(res["E"][0], res["dE"][0], cls) for cls, res in spec.items() if np.isfinite(res["E"][0])]
    return min(cand) if cand else (np.nan, np.nan, "-")

charge_gap, lowest_addition = {}, {}       # per run: (value, error); used again in section 8
rows = []
for name, run in runs.items():
    Ea, dEa, ca = lowest_level(spectra[name])
    lowest_addition[name] = (Ea, dEa)
    row = {"run": label(name), "lowest addition E/kappa": f"{Ea:.4f} ± {dEa:.4f}  (class {ca})"}
    if PH[name]:
        charge_gap[name] = (2 * Ea, 2 * dEa)
    else:
        Er, dEr, cr = lowest_level(removal_spectra[name])
        row["lowest removal E/kappa"] = f"{Er:.4f} ± {dEr:.4f}  (class {cr})"
        jk_sum = spectra[name][ca]["E_jk"][0] + removal_spectra[name][cr]["E_jk"][0]   # same bins: joint jackknife
        charge_gap[name] = (Ea + Er, pa.jk_error(jk_sum))
    free_gap = np.min(run.eps[run.eps > 0]) - np.max(run.eps[run.eps < 0])
    rows.append({**row, "charge gap / kappa": "{:.4f} ± {:.4f}".format(*charge_gap[name]),
                 "free gap / kappa": round(free_gap, 4)})
pd.DataFrame(rows).set_index("run")

### Correlator matrices

The matrices that enter the GEVP: for every symmetry class, $A_{ab}(\tau)$ of the class orbitals $a, b$ with
$\varepsilon > 0$, all runs in the same panels. Shown is $|A_{ab}(\tau)|$ on a log scale over the whole range
$0 \le \tau < \beta$: filled markers where $A_{ab} > 0$, open markers where $A_{ab} < 0$; time slices whose error is
larger than the value are left out. Errors: jackknife over the bins of `BINSIZE` configurations (for a mean, the
standard error of the bin averages). Dashed grey: the non-interacting correlator ($\phi = 0$), $e^{-\varepsilon_a
\tau}$ on the diagonal (up to the thermal backward part near $\tau = \beta$) and zero off the diagonal.

In [ ]:
run0 = next(iter(runs.values()))
for cls, series in onebody_corr.items():
    orbs = run0.classes[cls][pa.orbital_selection(run0, cls, "addition")]
    names = [f"orbital {o} (ε/κ = {run0.eps[o]:.3f})" for o in orbs]
    shown = names[:CORR_MAX_OPS]
    title = (rf"class {pa.sym_label(cls)}: addition correlators $A_{{ab}}(\tau)$;  "
             + ",  ".join(f"{i}: {n}" for i, n in enumerate(shown)))
    pa.correlator_grid(series, names, run0.delta, title, free=onebody_free.get(cls), ylabel="A", max_ops=CORR_MAX_OPS)
    plt.show()

## 6. Equal-time spin and charge correlations

From $G(\tau,\tau)$ of every configuration (Wick's theorem per time slice, averaged over the time slices): total charge
$\langle Q\rangle$ (0 at half filling), total spin $\langle \mathbf S^2\rangle$ (0 for a singlet ground state; at
$\mu = 0$ thermally excited charged states contribute $\sim e^{-\beta E_{\rm add}}$), local moment
$\langle s_x^2\rangle$ and double occupancy $\langle n_{x\uparrow} n_{x\downarrow}\rangle$ (site averages), and the
staggered spin and charge structure factors $S_{\rm AF} = \frac1{N_x}\sum_{xy} e_x e_y \langle s_x s_y\rangle$,
$S_{\rm CDW} = \frac1{N_x}\sum_{xy} e_x e_y \langle q_x q_y\rangle$ ($e_x = \pm1$ the sublattice sign). The plots show
the correlations averaged over site pairs at equal distance.

In [ ]:
eqt, rows = {}, []
for name, run in runs.items():
    G, ss, qq = run.equaltime(BINSIZE)
    obs = pa.equaltime_observables(G, ss, qq, run.sublattice)
    eqt[name] = obs
    me = lambda x: (x.mean(axis=0), x.std(axis=0, ddof=1) / np.sqrt(len(x)))
    row = dict(run=label(name))
    for key, x in (("Q", obs["Q"]), ("S^2", obs["S2"]), ("<s_x^2>", obs["m2"].mean(axis=1)),
                   ("<n_up n_down>", obs["d"].mean(axis=1)), ("S_AF", obs.get("S_AF")), ("S_CDW", obs.get("S_CDW"))):
        if x is not None:
            m, e = me(x)
            row[key] = f"{m:.5f} ± {e:.5f}"
    rows.append(row)
display(pd.DataFrame(rows).set_index("run"))

r, shells = pa.distance_shells(next(iter(runs.values())).positions)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for ax, key, title in zip(axes, ("ss", "qq"), (r"spin $\langle s_x s_y\rangle$", r"charge $\langle q_x q_y\rangle$")):
    for name in runs:
        sh = pa.shell_average(eqt[name][key], r, shells)
        ax.errorbar(shells, sh.mean(axis=0), sh.std(axis=0, ddof=1) / np.sqrt(len(sh)), fmt="o-", ms=3,
                    color=color[name], label=label(name))
    ax.axhline(0, color="#8a8a85", lw=0.8)
    ax.set_xlabel("distance (length unit of the input)")
    ax.set_title(title + ", averaged over pairs at equal distance")
axes[0].set_ylabel("correlation")
axes[0].legend()
plt.show()

## 7. Heavy tails

Near a zero of the fermion determinant (at a distance $d$ in field space) the weight vanishes like $d^2$ and the
propagator has an eigenvalue $\propto 1/d$. The expected consequence: an observable linear in $G$ has a distribution
tail $P(|O| > y) \sim y^{-3}$ (finite variance); an observable quadratic in $G$, such as the equal-time spin and
charge correlations of section 6 and every two-body correlator of section 8, has a tail $\sim y^{-3/2}$: its mean
exists, its variance does not. Jackknife errors of such an observable are unreliable, and a few configurations can
move its mean by many standard deviations. The first production (radial updates ineffective) showed this pattern
in the Hubbard run: tail exponents near 1 for $S_{\rm CDW}$, $S_{\rm AF}$ and the double occupancy, 3 to 6 for the
LUMO correlator.

For every observable: the samples as stored (configurations, or bins of a file reduced with `--bin`; averages over
bins keep a tail exponent below 2), their deviation from the median in units of the interquartile range (IQR), the
largest deviation and where it occurs, and the tail exponent $\alpha$ of $P({\rm dev} > y) \sim y^{-\alpha}$ (Hill
estimate from the largest 2 % of the deviations; rough, since consecutive configurations are correlated).
$\alpha < 2$: the variance is probably infinite. The plot shows the fraction of samples beyond a deviation $y$ on
log-log axes, with the slopes $-3$ and $-3/2$ for reference. (The site-averaged double occupancy is not listed:
at half filling it is $(1 - \langle s_x^2\rangle)/2$ on every configuration, with the same tail.)

In [ ]:
TAIL_OBS = [(PROBE_KEY, "linear"), ("<s_x^2>", "quadratic"), ("S_AF", "quadratic"), ("S_CDW", "quadratic")]
TAIL_COLORS = pa.COLORS[3:8]
TAIL_TEX = {PROBE_KEY: f"$A_{{\\rm LUMO}}(t={T_PROBE})$  (linear)", "<s_x^2>": r"$\langle s_x^2\rangle$",
            "<n_up n_down>": r"$\langle n_\uparrow n_\downarrow\rangle$", "S_AF": r"$S_{\rm AF}$", "S_CDW": r"$S_{\rm CDW}$"}

def where(chain, j, per_bin):
    # sample j of a chain: a configuration, or a bin of the reduced file
    if not per_bin or chain.bin == 1:
        return f"{chain.tag[-3:]}, configuration {chain.ids[j]}"
    p = (chain.skip + j) * chain.bin - chain.discard
    return f"{chain.tag[-3:]}, configurations {chain.ids[p]}-{chain.ids[min(p + chain.bin, len(chain.ids)) - 1]}"

def locate(counts, i):
    # (chain index, index in the chain) of the i-th of the concatenated samples
    k = int(np.searchsorted(np.cumsum(counts), i, side="right"))
    return k, i - int(np.sum(counts[:k]))

def tail_verdict(alpha):
    return "-" if not np.isfinite(alpha) else ("probably infinite" if alpha < 2 else "finite")

tail_samples, rows = {}, []
for name, run in runs.items():
    per = {k: [] for k, _ in TAIL_OBS}
    for c, h in zip(run.chains, histories[name]):
        per[PROBE_KEY].append(h[PROBE_KEY])
        o = pa.equaltime_observables(*c.equaltime(c.bin), run.sublattice)
        per["<s_x^2>"].append(o["m2"].mean(axis=1))
        if "S_AF" in o:
            per["S_AF"].append(o["S_AF"])
            per["S_CDW"].append(o["S_CDW"])
    tail_samples[name] = {}
    for key, kind in TAIL_OBS:
        if not per[key]:
            continue
        x, counts = np.concatenate(per[key]), np.array([len(v) for v in per[key]])
        st = pa.tail_stats(x)
        tail_samples[name][key] = x
        per_bin = key != PROBE_KEY
        at = where(run.chains[locate(counts, st["argmax"])[0]], locate(counts, st["argmax"])[1], per_bin) \
            if st["argmax"] >= 0 else "-"
        b = run.chains[0].bin
        unit = "configurations" if not per_bin or b == 1 else f"bins of {b} configurations"
        rows.append({"run": label(name), "observable": key, "in G": kind, "samples": f"{st['n']} {unit}",
                     "largest deviation / IQR": st["max_dev"], "at": at, "tail exponent alpha": st["alpha"],
                     "variance": tail_verdict(st["alpha"])})
display(pd.DataFrame(rows).set_index(["run", "observable"]))

def reference_slopes(ax, ymax, pmin, y0=3.0, p0=0.02):
    # lines P = p0 (y / y0)^-a from y0 to where they leave the plot (y = ymax or P = pmin), labelled at the end
    ax.set_ylim(pmin, 1.5)
    for a, ls in ((3.0, ":"), (1.5, "--")):
        y_end = min(ymax, y0 * (p0 / pmin) ** (1 / a))
        if y_end <= y0:
            continue
        y = np.geomspace(y0, y_end, 20)
        ax.plot(y, p0 * (y / y0) ** -a, color="#5c5c57", lw=0.9, ls=ls)
        if y_end >= 2 * y0:
            ax.text(y[-1], p0 * (y[-1] / y0) ** -a, f" slope -{a:g}", fontsize=7, color="#5c5c57", va="bottom")

fig, axes = plt.subplots(1, len(runs), figsize=(3.9 * len(runs), 3.3), sharey=True, squeeze=False,
                         constrained_layout=True)
for ax, (name, samples) in zip(axes[0], tail_samples.items()):
    ymax = 10.0
    for j, (key, x) in enumerate(samples.items()):
        y, P = pa.tail_curve(x)
        ok = y > 0.25
        ax.plot(y[ok], P[ok], color=TAIL_COLORS[j % len(TAIL_COLORS)], lw=1.3, label=TAIL_TEX.get(key, key))
        ymax = max(ymax, y[-1])
    reference_slopes(ax, ymax, 0.3 / max(len(x) for x in samples.values()))
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(label(name))
    ax.set_xlabel("deviation from the median / IQR")
axes[0, 0].set_ylabel("fraction of samples beyond")
axes[0, 0].legend(fontsize=7)
plt.show()

## 8. Two-body correlators: excitons and pairs

From stage 2 (`ppp_reduce2.py`): operator bases of orbital pairs from the window of orbitals around the gap, per
symmetry sector (product of the classes of the two orbitals, i.e. the symmetry of the state relative to the ground
state):

* **excitons** $O^\dagger = c^\dagger_c c_v$ (conduction $c$, valence $v$), spin triplet (connected part only) and
  singlet (connected $+\,2\times$ disconnected, minus $2\langle O\rangle\langle O^\dagger\rangle$), in combinations of
  definite particle-hole parity $C$ (spin-symmetric conjugation; $C = (-1)^I$ in the Hubbard model, where $I$ is the
  pseudospin of the notes): $(c,v)$ and $(\bar v, \bar c)$ (partner orbitals) combined with both signs;
* **pp pairs** $c^\dagger_{a\uparrow} c^\dagger_{b\downarrow}$, spin singlet and triplet ($a, b$ conduction). hh pairs are
  the particle-hole images (identical correlators on every configuration).

Energies $E/\kappa$ relative to the $N$-electron ground state, from the GEVP as in section 5 (plots up to
$\tau = \beta/2$, beyond which the backward propagating states take over). At $\tau = 0$ every correlator is the
equal-time value $\langle O_i O_j^\dagger\rangle$ in operator order: there a contraction of a sink $c^\dagger$
with a source $c$ is $-\langle c^\dagger c\rangle = G(t,t) - 1$, not $M^{-1}[t,t] = \langle c\,c^\dagger\rangle$.
The $-1$ (the contact term) adds $\delta_{v_i v_j}\,G_{c_i c_j}(t,t)$ to the exciton correlators; pp correlators
have none. Stage-2 files of version 2 contain it; for older ones it is added from the stage-1 file of the same chain
(same configurations and source times). Without such a file the exciton $\tau = 0$ is left out. A dipole-allowed (bright) singlet has $C = -1$ and the symmetry of a position
component: sector `-+` ($x$) or `+-` ($y$). Exciton binding energy: charge gap (section 5) minus the exciton energy.
In the Hubbard model, pseudospin symmetry makes the $C=-1$ singlet of the HOMO-LUMO pair degenerate with the
LUMO$^2$ pp pair; the last plot of this section tests this (ratio 1), and shows how far the long-range interaction
moves them apart.

In [ ]:
TWO_BODY_DIR = os.environ.get("PPP_REDUCED2", DATA_DIR.rstrip("/") + "2")   # stage-2 files <run>_cNN.h5
T0_2B, TD_2B = 2, 4          # GEVP time slices for the two-body correlators
FIT_WINDOW_2B = (4, 32)      # time slices; cut per state where the relative error exceeds MAX_REL_ERR
CORR_BLOCKS_2B = None        # blocks shown as correlator matrices: None = all, or e.g. [("exciton singlet", "-+", -1)]

two = pa.load_two_body(TWO_BODY_DIR, RUN_ORDER, DISCARD, onebody_dir=DATA_DIR)
if not two:
    print(f"no stage-2 files in {TWO_BODY_DIR}: section 8 is skipped, section 9 shows the single-particle levels only")
T0_OK = {name: all(c.t0_valid for c in chains) for name, chains in two.items()}   # exciton tau = 0 usable
for name, chains in two.items():
    n_file = sum(c.contact_terms for c in chains)
    n_added = sum(c.t0_valid and not c.contact_terms for c in chains)
    print(f"{label(name)}: tau = 0 contact term in the stage-2 files of {n_file} chains, added from stage 1 for {n_added}"
          + ("" if T0_OK[name] else f", missing for {len(chains) - n_file - n_added} (exciton tau = 0 left out)"))
blocks = {}
corr2, ops2 = {}, {}         # per block: [(run label, colour, mean, error)]; (operator labels, free energies)
for name, chains in two.items():
    c0, blocks[name] = chains[0], {}
    for (ch, sec), pairs in c0.pairs.items():
        cases = []
        if ch == "exciton":
            pe = pa.pair_energies(pairs, c0.eps, "exciton")
            for spin in ("singlet", "triplet"):
                M, labs = pa.parity_transform(pairs, c0.partner, spin)
                C = pa.exciton_bins(chains, sec, spin, BINSIZE)
                cases += [((f"exciton {spin}", sec, P), np.einsum("ia,ntab,jb->ntij", Mp, C, Mp), labs[P],
                           pa.combination_energies(Mp, pe)) for P, Mp in M.items() if len(Mp)]
        else:
            cases.append(((ch.replace("_", " "), sec, 0), pa.pp_bins(chains, ch, sec, BINSIZE),
                          [f"({a},{b})" for a, b in pairs], pa.pair_energies(pairs, c0.eps, ch)))
        for key, C, labs, free_e in cases:
            mean = C.mean(axis=0)
            if key[0].startswith("exciton") and not T0_OK[name]:
                mean[0] = np.nan                              # tau = 0 without the contact term
            corr2.setdefault(key, []).append((label(name), color[name], mean, C.std(axis=0, ddof=1) / np.sqrt(len(C))))
            ops2[key] = (labs, free_e)
            try:
                res = pa.state_energies(C, c0.delta, T0_2B, TD_2B, FIT_WINDOW_2B, None, MAX_REL_ERR)
                res["n_ops"] = C.shape[-1]
                blocks[name][key] = res
            except np.linalg.LinAlgError:
                print(f"{label(name)} {key}: C(T0) not positive definite, skipped")

rows = []
for name in blocks:
    for (ch, sec, P), res in sorted(blocks[name].items()):
        w = res["windows"][0]
        rows.append({"run": label(name), "channel": ch, "sector": sec, "C": {1: "+1", -1: "-1", 0: ""}[P],
                     "operators": res["n_ops"], "E0/kappa": res["E"][0], "error": res["dE"][0],
                     "fit window (tau)": f"{w[0] * runs[name].delta:.2f}-{w[1] * runs[name].delta:.2f}" if w else "-"})
if rows:
    display(pd.DataFrame(rows).set_index(["run", "channel", "sector", "C"]))

In [ ]:
def lowest(name, pred):
    cand = [(r["E"][0], r["dE"][0], k) for k, r in blocks[name].items() if pred(k) and np.isfinite(r["E"][0])]
    return min(cand) if cand else (np.nan, np.nan, None)

rows = []
for name in blocks:
    Ea, dEa = lowest_addition[name]
    gap, dgap = charge_gap[name]
    S, dS, kS = lowest(name, lambda k: k[0] == "exciton singlet" and k[2] == -1 and k[1] in ("-+", "+-"))
    T, dT, kT = lowest(name, lambda k: k[0] == "exciton triplet")
    P, dP, kP = lowest(name, lambda k: k[0] == "pp singlet")
    f = lambda E, dE: f"{E:.3f} ± {dE:.3f}"     # noqa: E731
    rows.append({"run": label(name), "charge gap": f(gap, dgap),
                 "bright singlet S": f(S, dS) + (f"  ({kS[1]})" if kS else ""),
                 "S binding (gap - S)": f(gap - S, np.hypot(dgap, dS)),
                 "lowest triplet T": f(T, dT) + (f"  ({kT[1]}, C={kT[2]:+d})" if kT else ""),
                 "S - T": f(S - T, np.hypot(dS, dT)),
                 "pp singlet - 2 E_add": f(P - 2 * Ea, np.hypot(dP, 2 * dEa))})
if rows:
    display(pd.DataFrame(rows).set_index("run").T)
    print("energies E/kappa; errors combined in quadrature (separate fits on the same bins); a joint jackknife follows when needed")

In [ ]:
keys = [("exciton singlet", "-+", -1, "bright singlet (sector − +, C = −1)"),
        ("exciton triplet", "-+", +1, "triplet (sector − +, C = +1)"),
        ("pp singlet", "++", 0, "pp singlet (sector + +)")]
for name in blocks:
    run = runs[name]
    fig, axes = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 3.0), sharey=True, constrained_layout=True)
    fig.suptitle(f"{label(name)}: two-body effective energies (GEVP states)", x=0.01, ha="left")
    for ax, (ch, sec, P, title) in zip(axes, keys):
        res = blocks[name].get((ch, sec, P))
        ax.set_title(title)
        ax.set_xlabel(TAU)
        if res is None:
            continue
        tau = np.arange(res["Eeff"].shape[1]) * run.delta
        tmax = 0
        for n in range(min(3, len(res["E"]))):
            good = np.isfinite(res["Eeff"][n]) & (res["Eeff_err"][n] < 0.5)
            if ch.startswith("exciton") and not T0_OK[name]:
                good[0] = False                               # tau = 0 without the contact term
            good[run.Nt // 2:] = False                        # beyond beta/2: backward propagation
            first = int(np.argmax(good)) if good.any() else 0
            last = first + (int(np.argmin(good[first:])) if not good[first:].all() else len(good) - first)
            ok = (np.arange(len(good)) >= first) & (np.arange(len(good)) < last)
            ax.errorbar(tau[ok], res["Eeff"][n, ok], res["Eeff_err"][n, ok], fmt="o", ms=2.5, lw=0.8,
                        color=pa.COLORS[n], label=f"state {n}")
            if res["windows"][n] is not None:
                ax.plot(np.array(res["windows"][n]) * run.delta, [res["E"][n]] * 2, color=pa.COLORS[n], lw=2.5)
            if ok.any():
                tmax = max(tmax, tau[ok].max())
        ax.set_xlim(0, min(1.15 * tmax + 0.5, run.beta / 2))     # beyond beta/2: backward propagation
        ax.set_ylim(0, 5)
    axes[0].set_ylabel(EKAPPA)
    axes[0].legend(fontsize=7)
    plt.show()

### Two-body correlator matrices

The matrices of the GEVP above, one figure per block (channel, sector and, for excitons, particle-hole parity $C$),
all runs in the same panels, drawn as in section 5: $|C_{ij}(\tau)|$ on a log scale, open markers for negative
values, time slices with an error larger than the value left out. Exciton operators are labelled by their pairs
(conduction, valence orbital), combined with the particle-hole partner pair where needed; pp operators by
$(a, b)$. `CORR_BLOCKS_2B` selects blocks, `CORR_MAX_OPS` limits the operators per block. At $\tau = 0$ the
values are the equal-time $\langle O_i O_j^\dagger\rangle$ (with the contact term, see above); for $\tau > \beta/2$ the
backward propagating states dominate.

In [ ]:
delta2 = next(iter(two.values()))[0].delta if two else None
for key in sorted(corr2):
    if CORR_BLOCKS_2B is not None and key not in CORR_BLOCKS_2B:
        continue
    ch, sec, P = key
    labs, free_e = ops2[key]
    head = f"{ch}, sector {pa.sym_label(sec)}" + (f", C = {P:+d}" if P else "") + ";  operators (free energy E/κ):  "
    ops = ";  ".join(f"{i}: {l} ({e:.3f})" for i, (l, e) in enumerate(zip(labs[:CORR_MAX_OPS], free_e)))
    pa.correlator_grid(corr2[key], labs, delta2, head + ops, ylabel="C", max_ops=CORR_MAX_OPS)
    plt.show()

### Heavy tails of the two-body correlators

As in section 7, for the diagonal correlators $C_{ii}(t)$ at the time slice `T_PROBE` of every operator of every
block, per stored bin of the stage-2 files. The table gives, for every block, the operator with the smallest tail
exponent and the number of operators with $\alpha < 2$; the plot the survival curve of the worst operator of every
channel.

In [ ]:
rows, worst2 = [], {}
for name, chains in two.items():
    c0 = chains[0]
    counts = np.array([c.nbins for c in chains])

    def at2(i):
        k, j = locate(counts, i)
        c = chains[k]
        first = int(c.first_id[c.skip + j])
        return f"{c.tag[-3:]}, " + (f"configurations {first}-{first + c.bin - 1}" if c.bin > 1 else f"configuration {first}")

    for (ch, sec), pairs in c0.pairs.items():
        items = []
        if ch == "exciton":
            for spin in ("singlet", "triplet"):
                M, labs = pa.parity_transform(pairs, c0.partner, spin)
                C = pa.exciton_bins(chains, sec, spin, c0.bin, t=T_PROBE)
                items += [((f"exciton {spin}", sec, P), np.einsum("ia,nab,ib->ni", Mp, C, Mp), labs[P])
                          for P, Mp in M.items() if len(Mp)]
        else:
            C = pa.pp_bins(chains, ch, sec, c0.bin, t=T_PROBE)
            items.append(((ch.replace("_", " "), sec, 0), np.einsum("nii->ni", C), [f"({a},{b})" for a, b in pairs]))
        for key, diag, labs in items:
            stats = [pa.tail_stats(diag[:, i]) for i in range(diag.shape[1])]
            alphas = np.array([st["alpha"] for st in stats])
            i = int(np.nanargmin(alphas)) if np.isfinite(alphas).any() else 0
            st = stats[i]
            unit = "configurations" if c0.bin == 1 else f"bins of {c0.bin} configurations"
            rows.append({"run": label(name), "channel": key[0], "sector": key[1], "C": {1: "+1", -1: "-1", 0: ""}[key[2]],
                         "samples": f"{st['n']} {unit}", "worst operator": labs[i],
                         "largest deviation / IQR": st["max_dev"], "at": at2(st["argmax"]) if st["argmax"] >= 0 else "-",
                         "tail exponent alpha": st["alpha"],
                         "operators with alpha < 2": f"{int(np.sum(alphas < 2))}/{len(alphas)}"})
            if np.isfinite(st["alpha"]) and st["alpha"] < worst2.get((name, key[0]), (np.inf,))[0]:
                worst2[(name, key[0])] = (st["alpha"], diag[:, i], f"{key[0]} {pa.sym_label(key[1])} {labs[i]}")
if rows:
    display(pd.DataFrame(rows).set_index(["run", "channel", "sector", "C"]))
    fig, axes = plt.subplots(1, len(two), figsize=(3.9 * len(two), 3.3), sharey=True, squeeze=False,
                             constrained_layout=True)
    for ax, name in zip(axes[0], two):
        ymax = 10.0
        for j, chan in enumerate(("exciton singlet", "exciton triplet", "pp singlet", "pp triplet")):
            if (name, chan) not in worst2:
                continue
            a, x, what = worst2[(name, chan)]
            y, P = pa.tail_curve(x)
            ok = y > 0.25
            ax.plot(y[ok], P[ok], color=TAIL_COLORS[j], lw=1.3, label=what)
            ymax = max(ymax, y[-1])
        reference_slopes(ax, ymax, 0.3 / max([len(w[1]) for k, w in worst2.items() if k[0] == name] or [10]))
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title(f"{label(name)} (t = {T_PROBE})")
        ax.set_xlabel("deviation from the median / IQR")
        ax.legend(fontsize=6)
    axes[0, 0].set_ylabel("fraction of bins beyond")
    fig.suptitle("two-body correlators: the operator with the heaviest tail in every channel", x=0.01, ha="left")
    plt.show()

In [ ]:
# pseudospin test: C = -1 singlet of the HOMO-LUMO pair over the LUMO^2 pp singlet (1 in the Hubbard model at every
# tau, tau = 0 included: a test of the contact term there)
fig, ax = plt.subplots(figsize=(6, 3.2), constrained_layout=True) if two else (None, None)
for name, chains in two.items():
    c0 = chains[0]
    lumo_o = int(np.argmin(np.where(c0.eps > 0, c0.eps, np.inf)))
    homo_o = int(c0.partner[lumo_o])
    sec_e = next(s for (ch, s), p in c0.pairs.items() if ch == "exciton" and any((a, b) == (lumo_o, homo_o) for a, b in p))
    sec_p = next(s for (ch, s), p in c0.pairs.items() if ch == "pp_singlet" and any((a, b) == (lumo_o, lumo_o) for a, b in p))
    ie = [i for i, (a, b) in enumerate(c0.pairs[("exciton", sec_e)]) if (a, b) == (lumo_o, homo_o)][0]
    ip = [i for i, (a, b) in enumerate(c0.pairs[("pp_singlet", sec_p)]) if (a, b) == (lumo_o, lumo_o)][0]
    Ce = pa.exciton_bins(chains, sec_e, "singlet", BINSIZE)[:, :, ie, ie]
    Cp = pa.pp_bins(chains, "pp_singlet", sec_p, BINSIZE)[:, :, ip, ip]
    ratio, dratio, _ = pa.jk_apply(lambda m: m[:, 0] / m[:, 1], np.stack([Ce, Cp], axis=-1))
    tau = np.arange(len(ratio)) * c0.delta
    good = np.isfinite(ratio) & (dratio < 0.5 * np.abs(ratio))
    t1 = 0 if T0_OK[name] else 1                               # tau = 0 needs the contact term
    good[:t1], good[len(ratio) // 2:] = False, False           # beyond beta/2: backward propagation
    last = t1 + (int(np.argmin(good[t1:])) if not good[t1:].all() else len(good) - t1)
    ok = (np.arange(len(ratio)) >= t1) & (np.arange(len(ratio)) < last)   # contiguous signal from t1
    ax.errorbar(tau[ok], ratio[ok], dratio[ok], fmt="o-", ms=3, lw=1, color=color[name], label=label(name))
if two:
    ax.axhline(1, color="#8a8a85", lw=0.8, ls="--")
    ax.set_xlabel(TAU)
    ax.set_ylabel(r"$C_{\rm exciton}(\tau)\,/\,C_{\rm pp}(\tau)$")
    ax.set_title("HOMO-LUMO singlet exciton (C = -1) / LUMO$^2$ pp singlet (pseudospin partners)")
    ax.legend()
    plt.show()

## 9. Level diagram

Energies $E/\kappa$ of the states relative to the $N$-electron ground state. The first figure compares the
single-particle levels of all runs; with stage-2 files, a figure per run follows that adds the two-body levels.
For every group, the non-interacting
energies are in the left column and the interacting ones (GEVP fits, sections 5 and 8) in the right column; dashed
lines join them in the order of the energies within a symmetry class or sector (state $n$ of a block with the
$n$-th lowest non-interacting level of its operators). Colours: symmetry class of the orbital (single particle),
sector of the pair (two-body). Levels closer than 2.5 % of the energy range are drawn side by side; shaded: $\pm$
error. Interacting levels with an error above `LEVEL_MAX_ERR` are left out.

* **single particle**: addition energies $E_{N+1} - E_N$ (non-interacting: $\varepsilon > 0$). With particle-hole
  symmetry ($\mu = 0$, bipartite lattice; section 5) the removal levels are their mirror image and are not drawn;
  without it the removal levels $-(E_{N-1} - E_N)$ are drawn below zero (non-interacting: $\varepsilon < 0$).
* **excitons** ($N$ electrons; non-interacting $\varepsilon_c - \varepsilon_v$) by spin and particle-hole parity
  $C$, and **pp pairs** ($N + 2$ electrons; non-interacting $\varepsilon_a + \varepsilon_b$). Only the pairs of the
  stage-2 orbital window appear, so higher non-interacting pair levels are missing.

In [ ]:
from matplotlib.patches import Patch
SYM_COLORS = {s: pa.COLORS[4 + i] for i, s in enumerate(pa.SYMMETRY_ORDER)}
GROUPS_2B = [("exciton singlet", -1, "exciton singlet\nC = -1"), ("exciton singlet", +1, "exciton singlet\nC = +1"),
             ("exciton triplet", +1, "exciton triplet\nC = +1"), ("exciton triplet", -1, "exciton triplet\nC = -1"),
             ("pp singlet", 0, "pp singlet\n(N + 2 electrons)"), ("pp triplet", 0, "pp triplet\n(N + 2 electrons)")]

def keep(E, dE):
    return bool(np.isfinite(E) and np.isfinite(dE) and dE <= LEVEL_MAX_ERR)

def single_particle_levels(name, run):
    # (non-interacting, interacting) levels: addition above zero; removal below zero only without particle-hole
    # symmetry (with it, the removal levels are the mirror image of the addition levels)
    free, inter = [], []
    for kind, spec, sign in (("addition", spectra[name], +1), ("removal", removal_spectra[name], -1)):
        for cls, res in spec.items():
            free += [(sign * e, cls, (kind, cls, n)) for n, e in enumerate(res["free"])]
            inter += [(sign * E, dE, cls, (kind, cls, n)) for n, (E, dE) in enumerate(zip(res["E"], res["dE"]))
                      if keep(E, dE)]
    return free, inter

def single_particle_title(name):
    return "single particle\naddition" if PH[name] else "single particle\naddition, removal"

def draw_levels(groups, title):
    energies = [lev[0] for g in groups for lev in g[1] + g[2]]
    fig, ax = plt.subplots(figsize=(1.9 * len(groups) + 2.2, 6.2), constrained_layout=True)
    ax.axhline(0, color="#8a8a85", lw=0.6)
    pa.level_diagram(ax, groups, SYM_COLORS, tol=0.025 * (max(energies) - min(energies)))
    ax.set_ylabel(r"$E/\kappa$  (relative to the $N$-electron ground state)")
    fig.suptitle(title, x=0.01, ha="left")
    ax.legend(handles=[Patch(color=SYM_COLORS[s], label=pa.sym_label(s)) for s in pa.SYMMETRY_ORDER], title="class / sector",
              loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8, title_fontsize=8)
    plt.show()

# all runs: single-particle levels
draw_levels([(label(name), *single_particle_levels(name, run)) for name, run in runs.items()],
            "single-particle levels, non-interacting and interacting: "
            + ("addition (removal = mirror image, particle-hole symmetry)" if all(PH.values())
               else "addition (E > 0); removal (E < 0) for the runs without particle-hole symmetry"))

# per run with the two-body levels (only with stage-2 files)
for name, run in runs.items():
    if not blocks.get(name):
        continue
    groups = [(single_particle_title(name), *single_particle_levels(name, run))]
    for ch, P, title in GROUPS_2B:
        f2, i2 = [], []
        for key, res in blocks.get(name, {}).items():
            if key[0] != ch or key[2] != P:
                continue
            f2 += [(e, key[1], (key, n)) for n, e in enumerate(np.sort(ops2[key][1]))]
            i2 += [(E, dE, key[1], (key, n)) for n, (E, dE) in enumerate(zip(res["E"], res["dE"])) if keep(E, dE)]
        if f2:
            groups.append((title, f2, i2))
    draw_levels(groups, f"{label(name)}: non-interacting and interacting levels")